# FireBIM — Exploring Digital Fire Regulations with RDF, SPARQL, and SHACL

This training notebook demonstrates how the FireBIM knowledge-graph approach can be used to **explore digital regulations, enrich them with thematic metadata, retrieve rules through natural-language questions, and validate building information with SHACL**.

The notebook is deliberately structured as a sequence of small, inspectable modules rather than a single application. Each module can be run independently and modified during a workshop or training session.

### Modules

1. **Exploring an RDF Graph and SPARQL** — loading and inspecting a FireBIM Regulation Graph and querying it with SPARQL.
2. **RDF + OWL: Ontology Loading, Thematic Enrichment, and Reasoning** — loading the FireBIM Regulation Ontology (FRO), linking statements to FireBIM Building Ontology (FBO) topics, and producing a small thematic metadata graph.
3. **Natural Language to SPARQL and Stakeholder-aware GraphRAG** — using an LLM to translate natural-language questions into SPARQL, retrieving the source regulations from the graph, and summarising the results for a selected stakeholder, task, and design phase.
4. **Validation** — applying a SHACL shape to building data with `pySHACL`.
5. **Chatbot** — wrapping the retrieval workflow in a small Gradio interface.

The notebook complements the reusable **Neuro-symbolic Requirement Generator**: the neuro-symbolic tool generates machine-interpretable requirements, while this notebook demonstrates how digital regulations can subsequently be explored and applied from a user perspective.

> **Important:** the LLM is used to generate queries and summaries; the regulatory text returned to the user is retrieved directly from the RDF graph. The generated SPARQL query is shown so that the interpretation can be inspected.

## Before you start

Upload the relevant data files to the Colab session, for example:

```text
RegulationGraph.ttl
FireRegulations_themes_only.ttl   # generated by Module 2
```

The ontology can be loaded directly from the FireBIM GitHub repository. A Gemini API key is required for the LLM-based parts of the notebook.

In [ ]:
# Install the notebook dependencies once.
%pip install -q rdflib pyvis networkx pyshacl pandas requests gradio

import json
import os
import re
import getpass
from pathlib import Path

import pandas as pd
import rdflib
import networkx as nx
from pyvis.network import Network
from IPython.display import display, HTML
from pyshacl import validate
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS, OWL, XSD

# Module 1 — Exploring an RDF Graph and SPARQL

The first module introduces the basic operations needed to work with a FireBIM Regulation Graph. The graph stores the structure of the regulatory document as RDF, including documents, chapters, articles, statements, text, references, and—when available—links to digital rules.

## Loading an RDF graph

In [ ]:
# Path to the regulation graph in the Colab environment.
graph_url = "/content/RegulationGraph.ttl"

g = Graph()
g.parse(graph_url, format="turtle")

print(f"The loaded graph contains {len(g):,} triples.")

## Exploring the graph

In [ ]:
# List the RDF classes that occur in the graph.
classes = sorted({str(o) for s, p, o in g.triples((None, RDF.type, None))})

print("Classes found in the graph:")
for cls in classes[:50]:
    print("-", cls)

### Visualising a small part of the graph

Large RDF graphs are difficult to display in full. The following cell visualises only the first few triples so that the graph structure can be inspected without creating an overly large figure.

In [ ]:
max_triples_to_visualize = 200

nx_graph = nx.DiGraph()
for i, (s, p, o) in enumerate(g):
    if i >= max_triples_to_visualize:
        break
    nx_graph.add_edge(str(s), str(o), label=str(p))

net = Network(
    height="600px",
    width="100%",
    notebook=True,
    cdn_resources="in_line",
)

for node in nx_graph.nodes:
    label = str(node).split("#")[-1].split("/")[-1][:40]
    net.add_node(node, label=label, title=node)

for u, v, data in nx_graph.edges(data=True):
    label = str(data["label"]).split("#")[-1].split("/")[-1][:30]
    net.add_edge(u, v, label=label, title=data["label"])

html_content = net.generate_html()
display(HTML(html_content))

## Querying the graph with SPARQL

SPARQL is used to retrieve information from the RDF graph. Start with a simple query over arbitrary triples, then move to queries that use the FireBIM Regulation Ontology.

In [ ]:
# Five arbitrary triples
query = """
SELECT ?s ?p ?o
WHERE {
    ?s ?p ?o .
}
LIMIT 5
"""

for row in g.query(query):
    print(row)

In [ ]:
# Query articles represented in the regulation graph.
query = """
PREFIX fro: <http://www.firebim.org/ontologies/fro#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?article
WHERE {
    ?article rdf:type fro:Article .
}
LIMIT 10
"""

results = g.query(query)
df_articles = pd.DataFrame(results, columns=[str(v) for v in results.vars])
df_articles

In [ ]:
# Query regulatory statements and their human-readable text.
query = """
PREFIX fro: <http://www.firebim.org/ontologies/fro#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?statement ?text
WHERE {
    ?statement rdf:type fro:Statement ;
               fro:hasText ?text .
}
LIMIT 20
"""

results = g.query(query)
df_statements = pd.DataFrame(
    [{str(v): str(row[v]) for v in results.vars} for row in results]
)
df_statements

# Module 2 — RDF + OWL: Ontology Loading and Thematic Enrichment

The second module introduces the FireBIM Regulation Ontology (FRO) and the FireBIM Building Ontology (FBO). It then adds thematic metadata to regulatory statements. This creates links such as:

```text
ex:Statement  fro:isAbout  fbo:FireCompartment
```

The thematic metadata can subsequently be used by natural-language retrieval to find rules based on domain concepts rather than only on document structure.

In [ ]:
FRO = Namespace("http://www.firebim.org/ontologies/fro#")
FBO = Namespace("http://www.firebim.org/ontologies/fbo#")
FBR = Namespace("http://www.firebim.org/regulations#")

fro_url = "https://raw.githubusercontent.com/AlexDonkers/ISBECodingCafe/refs/heads/main/data/fro.ttl"

fro_graph = Graph()
fro_graph.parse(fro_url, format="turtle")
print(f"FRO loaded with {len(fro_graph):,} triples.")

## Explore FRO classes

In [ ]:
query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT ?class ?label ?description
WHERE {
    ?class rdf:type owl:Class ;
           rdfs:label ?label ;
           rdfs:comment ?description .
    FILTER(LANG(?label) = "en" || LANG(?label) = "")
    FILTER(LANG(?description) = "en" || LANG(?description) = "")
}
LIMIT 100
"""

results = fro_graph.query(query)
df_classes = pd.DataFrame(
    [{str(v): str(row[v]) for v in results.vars} for row in results]
)
df_classes

## Add topics to regulatory statements

The following prototype uses Gemini to classify each regulatory statement into one of a small set of manually selected themes. In the current example, the classification score is stored explicitly in the output graph. In a larger implementation, the theme vocabulary can be taken from the FBO or another controlled domain vocabulary.

In [ ]:
# Gemini configuration.
GEMINI_MODEL = "gemini-2.5-flash"
API_KEY = os.getenv("GEMINI_API_KEY") or getpass.getpass("Enter your Gemini API key: ")
API_URL = f"https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent?key={API_KEY}"

# Topics to assign to regulation statements.
THEMES = {
    "Fire compartment": FBO["FireCompartment"],
    "Evacuation route": FBO["EvacuationRoute"],
}

regulation_graph = Graph()
regulation_graph.parse("/content/RegulationGraph.ttl", format="turtle")

query = """
PREFIX fro: <http://www.firebim.org/ontologies/fro#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?statement ?text
WHERE {
    ?statement rdf:type fro:Statement ;
               fro:hasText ?text .
}
"""

statement_results = list(regulation_graph.query(query))
texts = [
    {"id": i, "uri": str(row.statement), "text": str(row.text)}
    for i, row in enumerate(statement_results)
]

print(f"Found {len(texts)} statements with text.")

In [ ]:
def extract_json_array(text: str):
    """Extract a JSON array from a plain or fenced LLM response."""
    text = text.strip()
    text = re.sub(r"^```(?:json)?", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"```$", "", text).strip()

    start = text.find("[")
    end = text.rfind("]")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("No JSON array found in model response.")

    return json.loads(text[start:end + 1])


def gemini_generate(prompt: str) -> str:
    """Call Gemini using the REST API and return the generated text."""
    response = requests.post(
        API_URL,
        headers={"Content-Type": "application/json"},
        json={"contents": [{"parts": [{"text": prompt}]}]},
        timeout=120,
    )
    response.raise_for_status()
    data = response.json()

    try:
        return data["candidates"][0]["content"]["parts"][0]["text"]
    except (KeyError, IndexError) as exc:
        raise RuntimeError(f"Unexpected Gemini response: {data}") from exc

In [ ]:
# Keep the teaching example manageable by sending a limited number of statements.
MAX_STATEMENTS = 50
texts_for_classification = texts[:MAX_STATEMENTS]

topic_list = ", ".join(THEMES.keys())
prompt = f"""
You are a fire safety expert.

Classify each text into ZERO or ONE of the following themes:
{topic_list}

Return pure JSON with this structure:
[
  {{"id": 0, "theme": "Fire compartment", "confidence": 0.92}},
  {{"id": 1, "theme": null, "confidence": 0.12}}
]

Use null when none of the themes is sufficiently relevant.
Do not add explanations or markdown formatting.

Input texts:
{json.dumps([{'id': t['id'], 'text': t['text']} for t in texts_for_classification], indent=2)}
"""

response_text = gemini_generate(prompt)
classifications = extract_json_array(response_text)
print(json.dumps(classifications[:5], indent=2))

In [ ]:
# Create a small graph containing only the thematic annotations.
new_g = Graph()
new_g.bind("fro", FRO)
new_g.bind("fbo", FBO)

confidence_threshold = 0.8

for item in classifications:
    item_id = int(item["id"])
    theme_label = item.get("theme")
    confidence = float(item.get("confidence", 0))

    if not (0 <= item_id < len(texts_for_classification)):
        continue

    statement_uri = URIRef(texts_for_classification[item_id]["uri"])

    if theme_label in THEMES and confidence >= confidence_threshold:
        theme_uri = THEMES[theme_label]
        new_g.add((statement_uri, FRO["isAbout"], theme_uri))
        new_g.add(
            (statement_uri, FRO["confidence"], Literal(confidence, datatype=XSD.decimal))
        )
        print(f"{statement_uri.split('#')[-1]} → {theme_label} ({confidence:.2f})")
    else:
        print(f"Skipped {statement_uri.split('#')[-1]} (confidence={confidence:.2f})")

output_theme_file = "/content/FireRegulations_themes_only.ttl"
new_g.serialize(destination=output_theme_file, format="turtle")
print(f"Saved {len(new_g)} thematic triples to {output_theme_file}")

### Result

The thematic output is a separate graph so that it can be merged with the regulation graph when needed without modifying the original regulatory content. In the next module, this graph is loaded together with the regulation graph and queried through the `fro:isAbout` relation.

# Module 3 — Natural Language to SPARQL using LLMs

The objective is to let users query digital regulations without having to write SPARQL manually. The LLM receives a compact semantic context and a small set of example question/query pairs. It generates the **query**, while the regulation graph remains the source of the returned regulatory content.

In [ ]:
# Load the regulation graph and the thematic graph separately.
regulation_graph = Graph()
regulation_graph.parse("/content/RegulationGraph.ttl", format="turtle")

# This is the graph produced by Module 2. If the file is not available,
# the workflow still works on the regulation graph alone.
themes_graph = Graph()
if Path("/content/FireRegulations_themes_only.ttl").exists():
    themes_graph.parse("/content/FireRegulations_themes_only.ttl", format="turtle")

# Combine them for local GraphRAG demonstrations.
g = regulation_graph + themes_graph
print(f"Loaded {len(g):,} triples from regulation + thematic graphs.")

## Prepare ontology context

In [ ]:
def get_label(graph: Graph, uri):
    for lbl in graph.objects(uri, RDFS.label):
        if isinstance(lbl, Literal) and (lbl.language in (None, "en")):
            return str(lbl)
    return str(uri).split("#")[-1].split("/")[-1]


def get_comment(graph: Graph, uri):
    for cmt in graph.objects(uri, RDFS.comment):
        if isinstance(cmt, Literal) and (cmt.language in (None, "en")):
            return str(cmt)
    return ""

class_info = []
for cls in fro_graph.subjects(RDF.type, OWL.Class):
    class_info.append(
        f"- {get_label(fro_graph, cls)} ({cls}): {get_comment(fro_graph, cls)}"
    )

property_info = []
for prop in fro_graph.subjects(RDF.type, OWL.ObjectProperty):
    property_info.append(
        f"- {get_label(fro_graph, prop)} ({prop}): {get_comment(fro_graph, prop)}"
    )
for prop in fro_graph.subjects(RDF.type, OWL.DatatypeProperty):
    property_info.append(
        f"- {get_label(fro_graph, prop)} ({prop}): {get_comment(fro_graph, prop)}"
    )

MAX_ONTOLOGY_ITEMS = 100
ontology_summary = (
    "FRO CLASSES:\n" + "\n".join(class_info[:MAX_ONTOLOGY_ITEMS]) +
    "\n\nFRO PROPERTIES:\n" + "\n".join(property_info[:MAX_ONTOLOGY_ITEMS]) +
    "\n\nFBO examples:\n- fbo:FireCompartment\n- fbo:EvacuationRoute\n"
)

print(ontology_summary[:4000], "...")

## Natural language → SPARQL

In [ ]:
FEW_SHOT_EXAMPLES = """
EXAMPLE QUESTION 1: all rules about fire compartments
EXAMPLE ANSWER 1:
SELECT DISTINCT ?rule WHERE {
    ?rule fro:isAbout ?topic .
    ?topic rdfs:subClassOf* fbo:FireCompartment .
}

EXAMPLE QUESTION 2: all rules about fire compartments and their text
EXAMPLE ANSWER 2:
SELECT DISTINCT ?rule ?text WHERE {
    ?rule fro:isAbout ?topic .
    ?topic rdfs:subClassOf* fbo:FireCompartment .
    ?rule fro:hasText ?text .
}

EXAMPLE QUESTION 3: all rules about fire compartments and residential functions
EXAMPLE ANSWER 3:
SELECT DISTINCT ?rule WHERE {
    ?rule fro:isAbout ?topic1, ?topic2 .
    ?topic1 rdfs:subClassOf* fbo:FireCompartment .
    ?topic2 rdfs:subClassOf* fbo:ResidentialFunction .
}

EXAMPLE QUESTION 4: all rules about fire compartments or residential functions
EXAMPLE ANSWER 4:
SELECT DISTINCT ?rule WHERE {
    ?rule fro:isAbout ?topic .
    {
        ?topic rdfs:subClassOf* fbo:FireCompartment .
    }
    UNION
    {
        ?topic rdfs:subClassOf* fbo:ResidentialFunction .
    }
}
"""


def nl_to_sparql(question: str) -> str:
    prompt = f"""
You are an RDF and SPARQL expert working with the FireBIM knowledge graph.

Use only the vocabulary and patterns supported by the context below.
Use `fro:hasText` for the text of regulatory statements.
Use `fro:isAbout` for thematic links.

{ontology_summary}

{FEW_SHOT_EXAMPLES}

Convert the following natural-language question into a READ-ONLY SPARQL SELECT query.
Do not generate INSERT, DELETE, CONSTRUCT, UPDATE, SERVICE, or other write operations.
Return only SPARQL code.

QUESTION:
{question}
"""

    query_text = gemini_generate(prompt)
    query_text = re.sub(r"^```(?:sparql)?", "", query_text.strip(), flags=re.IGNORECASE)
    query_text = re.sub(r"```$", "", query_text.strip())
    return query_text.strip()

In [ ]:
question = "All rules about fire compartments and the text of the rules."
sparql_query = nl_to_sparql(question)

print("Generated SPARQL query:\n")
print(sparql_query)

In [ ]:
# Execute the generated SELECT query directly against the RDF graph.
results = g.query(sparql_query)
rows = [{str(var): str(value) for var, value in row.asdict().items()} for row in results]

df_results = pd.DataFrame(rows)
df_results.head(20)

## Stakeholder-, task-, and phase-aware summaries

The retrieval result can now be interpreted for a particular project context. The configuration below separates three dimensions:

- **Stakeholder** — who is using the information.
- **Task** — what that person is currently trying to accomplish.
- **Design phase** — where the project is in its lifecycle.

The role information is represented as RDF so that the same approach can later be connected to a richer project- or organisation-specific knowledge graph.

In [ ]:
CONSTRUCTION_ROLES_TTL = r'''
@prefix ex: <http://example.org/fireSafety#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .

ex:Stakeholder a rdfs:Class .
ex:Role a rdfs:Class .
ex:Responsibility a rdfs:Class .
ex:LegalResponsibility a rdfs:Class .
ex:CommunicationRelation a rdfs:Class .
ex:HandoverRelation a rdfs:Class .

ex:hasRole a rdf:Property ; rdfs:domain ex:Stakeholder ; rdfs:range ex:Role .
ex:hasResponsibility a rdf:Property ; rdfs:domain ex:Stakeholder ; rdfs:range ex:Responsibility .
ex:hasLegalResponsibility a rdf:Property ; rdfs:domain ex:Stakeholder ; rdfs:range ex:LegalResponsibility .
ex:communicatesWith a rdf:Property ; rdfs:domain ex:Stakeholder ; rdfs:range ex:Stakeholder .
ex:handsOverTo a rdf:Property ; rdfs:domain ex:Stakeholder ; rdfs:range ex:Stakeholder .

ex:Client a ex:Stakeholder ;
    foaf:name "Client / Building Owner" ;
    ex:hasRole [ a ex:Role ; rdfs:label "Commission the project, define functional and safety requirements, and provide funding." ] ;
    ex:hasResponsibility [ a ex:Responsibility ; rdfs:label "Set project objectives, ensure compliance with fire safety standards, and approve design stages." ] ;
    ex:hasLegalResponsibility [ a ex:LegalResponsibility ; rdfs:label "Under the Omgevingswet and Bbl, ensure building design and use are fire-safe, appoint competent parties, maintain fire safety post-construction." ] ;
    ex:communicatesWith ex:Architect, ex:FireEngineer, ex:ProjectManager ;
    ex:handsOverTo ex:Architect .

ex:Architect a ex:Stakeholder ;
    foaf:name "Architect" ;
    ex:hasRole [ a ex:Role ; rdfs:label "Lead designer responsible for overall building design and coordination of disciplines." ] ;
    ex:hasResponsibility [ a ex:Responsibility ; rdfs:label "Integrate fire safety principles into architectural layouts." ] ;
    ex:hasLegalResponsibility [ a ex:LegalResponsibility ; rdfs:label "Must design per Bbl fire-safety performance requirements and submit under the Omgevingswet." ] ;
    ex:communicatesWith ex:FireEngineer, ex:StructuralEngineer, ex:MEPEngineer, ex:ProjectManager ;
    ex:handsOverTo ex:Contractor .

ex:FireEngineer a ex:Stakeholder ;
    foaf:name "Fire Engineer / Fire Safety Consultant" ;
    ex:hasRole [ a ex:Role ; rdfs:label "Specialist ensuring that fire protection systems and design meet codes and performance criteria." ] ;
    ex:hasResponsibility [ a ex:Responsibility ; rdfs:label "Conduct fire risk analysis, evacuation modelling, and specify active/passive systems." ] ;
    ex:hasLegalResponsibility [ a ex:LegalResponsibility ; rdfs:label "Ensure compliance with Bbl performance requirements, prepare documentation for municipal review." ] ;
    ex:communicatesWith ex:Architect, ex:MEPEngineer, ex:BuildingControl, ex:FireService ;
    ex:handsOverTo ex:BuildingControl .

ex:StructuralEngineer a ex:Stakeholder ;
    foaf:name "Structural Engineer" ;
    ex:hasRole [ a ex:Role ; rdfs:label "Design structural systems capable of maintaining integrity during fire exposure." ] ;
    ex:hasResponsibility [ a ex:Responsibility ; rdfs:label "Ensure structural fire resistance and coordinate with the fire engineer." ] ;
    ex:hasLegalResponsibility [ a ex:LegalResponsibility ; rdfs:label "Design per Bbl structural safety in case of fire, meet NEN standards, and ensure soundness of design." ] ;
    ex:communicatesWith ex:FireEngineer, ex:Contractor ;
    ex:handsOverTo ex:Contractor .

ex:MEPEngineer a ex:Stakeholder ;
    foaf:name "Mechanical, Electrical & Plumbing Engineer" ;
    ex:hasRole [ a ex:Role ; rdfs:label "Design building services including HVAC, sprinklers, smoke extraction, alarms, and emergency power." ] ;
    ex:hasResponsibility [ a ex:Responsibility ; rdfs:label "Specify fire detection/suppression systems and integrate with BMS." ] ;
    ex:hasLegalResponsibility [ a ex:LegalResponsibility ; rdfs:label "Ensure systems meet Bbl and NEN standards for detection, suppression, and safety." ] ;
    ex:communicatesWith ex:FireEngineer, ex:Architect, ex:Contractor ;
    ex:handsOverTo ex:Contractor .

ex:ProjectManager a ex:Stakeholder ;
    foaf:name "Project Manager" ;
    ex:hasRole [ a ex:Role ; rdfs:label "Oversee design development, schedule, and budget, ensuring compliance with fire safety requirements." ] ;
    ex:hasResponsibility [ a ex:Responsibility ; rdfs:label "Coordinate between disciplines, track regulatory submissions, manage communication flow." ] ;
    ex:hasLegalResponsibility [ a ex:LegalResponsibility ; rdfs:label "Ensure all required permits and fire-safe use notifications under the Omgevingswet/Bbl are obtained." ] ;
    ex:communicatesWith ex:Client, ex:Architect, ex:Contractor, ex:BuildingControl ;
    ex:handsOverTo ex:BuildingControl .

ex:BuildingControl a ex:Stakeholder ;
    foaf:name "Building Control Officer / Authority Having Jurisdiction" ;
    ex:hasRole [ a ex:Role ; rdfs:label "Review and approve design for compliance with fire codes and standards." ] ;
    ex:hasResponsibility [ a ex:Responsibility ; rdfs:label "Inspect documentation, conduct site visits, issue approvals." ] ;
    ex:hasLegalResponsibility [ a ex:LegalResponsibility ; rdfs:label "Enforce compliance with Omgevingswet and Bbl; may refuse occupancy until compliance achieved." ] ;
    ex:communicatesWith ex:Architect, ex:FireEngineer, ex:ProjectManager ;
    ex:handsOverTo ex:FacilitiesManager .

ex:FireService a ex:Stakeholder ;
    foaf:name "Fire Service / Fire Brigade Liaison" ;
    ex:hasRole [ a ex:Role ; rdfs:label "Ensure firefighting access, water supply, and rescue features are adequate." ] ;
    ex:hasResponsibility [ a ex:Responsibility ; rdfs:label "Review access routes, hydrant placement, fire lifts, smoke-control provisions." ] ;
    ex:hasLegalResponsibility [ a ex:LegalResponsibility ; rdfs:label "Can inspect and enforce fire safety for public buildings and advise municipalities under Bbl." ] ;
    ex:communicatesWith ex:FireEngineer, ex:Architect, ex:FacilitiesManager ;
    ex:handsOverTo ex:FacilitiesManager .

ex:Contractor a ex:Stakeholder ;
    foaf:name "Contractor / Main Builder" ;
    ex:hasRole [ a ex:Role ; rdfs:label "Execute construction according to design specifications and fire safety standards." ] ;
    ex:hasResponsibility [ a ex:Responsibility ; rdfs:label "Implement passive fire protection and ensure correct installation of systems." ] ;
    ex:hasLegalResponsibility [ a ex:LegalResponsibility ; rdfs:label "Must comply with approved plans and Bbl requirements; responsible under Arbeidsomstandighedenwet for safe sites." ] ;
    ex:communicatesWith ex:Architect, ex:MEPEngineer, ex:Subcontractor, ex:ProjectManager ;
    ex:handsOverTo ex:FacilitiesManager .

ex:Subcontractor a ex:Stakeholder ;
    foaf:name "Subcontractors (Fire Protection Specialists)" ;
    ex:hasRole [ a ex:Role ; rdfs:label "Install fire detection, suppression, and compartmentation systems." ] ;
    ex:hasResponsibility [ a ex:Responsibility ; rdfs:label "Follow manufacturer and fire engineer specifications; test and commission systems." ] ;
    ex:hasLegalResponsibility [ a ex:LegalResponsibility ; rdfs:label "Meet Bbl performance requirements, install certified components, maintain documentation, liable under Dutch civil law for defects." ] ;
    ex:communicatesWith ex:Contractor, ex:MEPEngineer, ex:FireEngineer ;
    ex:handsOverTo ex:Contractor .

ex:QuantitySurveyor a ex:Stakeholder ;
    foaf:name "Quantity Surveyor / Cost Consultant" ;
    ex:hasRole [ a ex:Role ; rdfs:label "Manage project costs, including fire safety system budgets." ] ;
    ex:hasResponsibility [ a ex:Responsibility ; rdfs:label "Estimate costs for fire-resistant materials and safety features." ] ;
    ex:hasLegalResponsibility [ a ex:LegalResponsibility ; rdfs:label "Ensure cost planning allows compliance with Bbl requirements; liable if cost constraints compromise safety." ] ;
    ex:communicatesWith ex:Architect, ex:FireEngineer, ex:Client ;
    ex:handsOverTo ex:ProjectManager .

ex:FacilitiesManager a ex:Stakeholder ;
    foaf:name "Facilities Manager / End-User Representative" ;
    ex:hasRole [ a ex:Role ; rdfs:label "Responsible for building operation and maintenance after handover." ] ;
    ex:hasResponsibility [ a ex:Responsibility ; rdfs:label "Manage fire drills, maintenance of alarms/sprinklers, and update risk assessments." ] ;
    ex:hasLegalResponsibility [ a ex:LegalResponsibility ; rdfs:label "Under Dutch fire-safe use obligations, ensure building remains fire-safe and accessible for firefighting." ] ;
    ex:communicatesWith ex:FireService, ex:Contractor, ex:BuildingControl ;
    ex:handsOverTo ex:FireService .

ex:ProductManufacturer a ex:Stakeholder ;
    foaf:name "Product Manufacturers / Suppliers" ;
    ex:hasRole [ a ex:Role ; rdfs:label "Provide certified fire safety materials and systems." ] ;
    ex:hasResponsibility [ a ex:Responsibility ; rdfs:label "Supply tested and approved components compliant with design specifications." ] ;
    ex:hasLegalResponsibility [ a ex:LegalResponsibility ; rdfs:label "Ensure compliance with CE/Construction Products Regulation and Bbl performance criteria; liable under product liability law." ] ;
    ex:communicatesWith ex:Contractor, ex:FireEngineer ;
    ex:handsOverTo ex:Contractor .

ex:HealthSafetyOfficer a ex:Stakeholder ;
    foaf:name "Health & Safety Officer" ;
    ex:hasRole [ a ex:Role ; rdfs:label "Ensure worker and occupant safety during construction and post-occupancy." ] ;
    ex:hasResponsibility [ a ex:Responsibility ; rdfs:label "Oversee fire safety during construction and compliance with safety management regulations." ] ;
    ex:hasLegalResponsibility [ a ex:LegalResponsibility ; rdfs:label "Under the Arbeidsomstandighedenwet, must ensure fire risk management on construction sites." ] ;
    ex:communicatesWith ex:Contractor, ex:ProjectManager, ex:FireEngineer ;
    ex:handsOverTo ex:ProjectManager .

ex:CommissioningAgent a ex:Stakeholder ;
    foaf:name "Commissioning Agent / Independent Tester" ;
    ex:hasRole [ a ex:Role ; rdfs:label "Verify that installed fire safety systems perform as designed." ] ;
    ex:hasResponsibility [ a ex:Responsibility ; rdfs:label "Test alarms, sprinklers, smoke control systems, and evacuation lighting before handover." ] ;
    ex:hasLegalResponsibility [ a ex:LegalResponsibility ; rdfs:label "Verify installations per Bbl and NEN standards; liable under Dutch law for negligent certification." ] ;
    ex:communicatesWith ex:Contractor, ex:FireEngineer, ex:MEPEngineer ;
    ex:handsOverTo ex:FacilitiesManager .

ex:Client ex:handsOverTo ex:Architect .
ex:Architect ex:handsOverTo ex:Contractor .
ex:FireEngineer ex:handsOverTo ex:BuildingControl .
ex:Contractor ex:handsOverTo ex:FacilitiesManager .
ex:Subcontractor ex:handsOverTo ex:Contractor .
ex:CommissioningAgent ex:handsOverTo ex:FacilitiesManager .
ex:FacilitiesManager ex:handsOverTo ex:FireService .
'''

roles_graph = Graph()
roles_graph.parse(data=CONSTRUCTION_ROLES_TTL, format="turtle")
print(f"Loaded construction-role graph with {len(roles_graph):,} triples.")

In [ ]:
EX = Namespace("http://example.org/fireSafety#")
FOAF = Namespace("http://xmlns.com/foaf/0.1/")


def get_stakeholder_profile(role_name: str):
    """Return the role profile for a stakeholder by name."""
    role_name_lower = role_name.strip().lower()

    for stakeholder in roles_graph.subjects(RDF.type, EX.Stakeholder):
        names = [str(v) for v in roles_graph.objects(stakeholder, FOAF.name)]
        if any(role_name_lower in name.lower() for name in names):
            profile = {
                "name": names[0] if names else str(stakeholder).split("#")[-1],
                "role": [str(v) for v in roles_graph.objects(stakeholder, EX.hasRole)],
                "responsibility": [str(v) for v in roles_graph.objects(stakeholder, EX.hasResponsibility)],
                "legal_responsibility": [str(v) for v in roles_graph.objects(stakeholder, EX.hasLegalResponsibility)],
                "communicates_with": [
                    str(roles_graph.value(v, FOAF.name) or v).split("#")[-1]
                    for v in roles_graph.objects(stakeholder, EX.communicatesWith)
                ],
                "hands_over_to": [
                    str(roles_graph.value(v, FOAF.name) or v).split("#")[-1]
                    for v in roles_graph.objects(stakeholder, EX.handsOverTo)
                ],
            }
            return profile

    raise ValueError(f"Stakeholder not found: {role_name}")

### Configure the user context

In [ ]:
# Change these three values to adapt the final interpretation.
STAKEHOLDER = "Architect"
TASK = "coordinate the architectural layout and its fire-safety information"
DESIGN_PHASE = "early design phase"

stakeholder_profile = get_stakeholder_profile(STAKEHOLDER)
print(json.dumps(stakeholder_profile, indent=2))

In [ ]:
def stakeholder_aware_summary(rows, stakeholder: str, task: str = "", design_phase: str = "") -> str:
    """Summarise retrieved regulation results for a project context."""
    profile = get_stakeholder_profile(stakeholder)

    role_context = f"""
STAKEHOLDER:
{profile['name']}

ROLE:
{json.dumps(profile['role'], indent=2)}

RESPONSIBILITIES:
{json.dumps(profile['responsibility'], indent=2)}

LEGAL RESPONSIBILITIES:
{json.dumps(profile['legal_responsibility'], indent=2)}

COMMUNICATES WITH:
{json.dumps(profile['communicates_with'], indent=2)}

HANDOVER TO:
{json.dumps(profile['hands_over_to'], indent=2)}
"""

    task_context = task.strip() or "No specific task was provided."
    phase_context = design_phase.strip() or "No specific design phase was provided."

    prompt = f"""
You are supporting a stakeholder in a FireBIM building-project workflow.

The retrieved regulatory statements below are the authoritative content for this summary.
Do not invent additional legal requirements and do not present general professional knowledge
as if it were stated in the regulations. Clearly distinguish between:
1. what the retrieved regulatory statements say;
2. what this means for the selected stakeholder;
3. practical actions or coordination suggestions derived from that information.

USER CONTEXT
{role_context}

CURRENT TASK:
{task_context}

DESIGN PHASE:
{phase_context}

RETRIEVED REGULATION RESULTS:
{json.dumps(rows, indent=2, ensure_ascii=False)}

Return a concise structured answer with exactly these sections:
## Relevant requirements
## What this means for the stakeholder
## Actions for the current task
## Coordination and information to check

Mention the regulatory statements as the source of the requirements. Do not claim that the
role dataset itself constitutes legislation.
"""

    return gemini_generate(prompt).strip()


summary_text = stakeholder_aware_summary(
    rows,
    stakeholder=STAKEHOLDER,
    task=TASK,
    design_phase=DESIGN_PHASE,
)

print(summary_text)

### Why this is different from simply adding a stakeholder to the prompt

The stakeholder information is represented separately from the retrieved regulations. This keeps the responsibilities of the actor distinct from the legal content returned by the regulation graph. The three context variables can therefore be changed without changing the query-generation method.

For example:

```python
STAKEHOLDER = "Fire Engineer / Fire Safety Consultant"
TASK = "review the evacuation strategy"
DESIGN_PHASE = "concept design"
```

The same retrieved regulation set can then be reinterpreted for that context.

# Module 4 — SHACL validation

The retrieved digital rules are intended to be executable against building information represented as RDF. The following example uses `pySHACL` to validate a small building graph against a SHACL shape.

In [ ]:
data_graph_ttl = """
@prefix bot: <https://w3id.org/bot#> .
@prefix fbo-nl: <http://www.firebim.org/ontologies/fbo-nl#> .
@prefix ex: <http://example.org/data#> .

ex:FireCompartment_01 a fbo-nl:FireCompartment ;
    bot:containsZone ex:Living, ex:Bedroom, ex:Hall, ex:Staircase, ex:Toilet, ex:Bathroom, ex:Dining, ex:Kitchen .

ex:Living a bot:Space ; fbo-nl:isEnclosedSpace true .
ex:Bedroom a bot:Space ; fbo-nl:isEnclosedSpace true .
ex:Hall a bot:Space ; fbo-nl:isEnclosedSpace true .
ex:Staircase a bot:Space ; fbo-nl:isEnclosedSpace true .
ex:Toilet a bot:Space ; fbo-nl:isEnclosedSpace true .
ex:Bathroom a bot:Space ; fbo-nl:isEnclosedSpace true .
ex:Dining a bot:Space ; fbo-nl:isEnclosedSpace true .
ex:Kitchen a bot:Space ; fbo-nl:isEnclosedSpace true .

# Deliberate violation: the room is not contained in a fire compartment.
ex:Room a bot:Space ; fbo-nl:isEnclosedSpace true .
"""

shapes_graph_ttl = """
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix bot: <https://w3id.org/bot#> .
@prefix fbo-nl: <http://www.firebim.org/ontologies/fbo-nl#> .
@prefix fbr: <http://www.firebim.org/regulations#> .

fbr:EnclosedSpaceInFireCompartmentShape
    a sh:NodeShape ;
    sh:target [
        a sh:SPARQLTarget ;
        sh:prefixes [
            sh:declare [
                sh:prefix "bot" ;
                sh:namespace "https://w3id.org/bot#" ;
            ] , [
                sh:prefix "fbo-nl" ;
                sh:namespace "http://www.firebim.org/ontologies/fbo-nl#" ;
            ] , [
                sh:prefix "xsd" ;
                sh:namespace "http://www.w3.org/2001/XMLSchema#" ;
            ]
        ] ;
        sh:select "SELECT ?this WHERE { ?this a <https://w3id.org/bot#Space> . ?this <http://www.firebim.org/ontologies/fbo-nl#isEnclosedSpace> ?val . FILTER(xsd:boolean(?val) = true) }" ;
    ] ;
    sh:property [
        sh:path [ sh:inversePath bot:containsZone ] ;
        sh:class fbo-nl:FireCompartment ;
        sh:minCount 1 ;
        sh:message "An enclosed space must be contained within a fire compartment." ;
    ] .
"""

print("Building and shape graphs...")

In [ ]:
conforms, results_graph, results_text = validate(
    data_graph_ttl,
    shacl_graph=shapes_graph_ttl,
    data_graph_format="turtle",
    shacl_graph_format="turtle",
    inference="rdfs",
    meta_shacl=True,
)

print(f"Validation result: {'CONFORMS' if conforms else 'VIOLATION FOUND'}")
print("\n--- SHACL validation report ---")
print(results_text)

# Module 5 — A small FireBIM chatbot

The final module wraps the retrieval and validation workflow in a minimal Gradio interface. The example below uses the **real natural-language-to-SPARQL function from Module 3** and executes the generated query locally against the RDF graph.

In [ ]:
def execute_sparql(query: str):
    """Execute a read-only SPARQL query against the local graph."""
    results = g.query(query)
    return [
        {str(var): str(value) for var, value in row.asdict().items()}
        for row in results
    ]


def chatbot_response(message, history):
    """Generate a SPARQL query, execute it, and return the retrieved results."""
    try:
        sparql = nl_to_sparql(message)
        results = execute_sparql(sparql)

        if results:
            result_table = pd.DataFrame(results).head(20).to_markdown(index=False)
        else:
            result_table = "No matching results were returned."

        return (
            "### Generated SPARQL\n\n"
            f"```sparql\n{sparql}\n```\n\n"
            "### Retrieved regulations\n\n"
            f"{result_table}"
        )
    except Exception as exc:
        return f"**Error:** `{exc}`"

In [ ]:
# Launch the chatbot.
# Run this cell in Colab/local Jupyter to start the interface.

demo = gr.ChatInterface(
    fn=chatbot_response,
    title="FireBIM Regulation Chatbot",
    description="Ask a natural-language question about the digital fire regulations.",
)

demo.launch()

# Further reading

The work demonstrated in this notebook is related to the following FireBIM publications:

- [Retrieval of Digital Regulations using Neuro-Symbolic AI with GraphRAG](https://pure.tue.nl/ws/portalfiles/portal/370207675/DonkersAJA_CameraReady.pdf)
- [A Hybrid BERT–LLM Approach for Regulation Graph Generation & Visualization from Fire Safety Documents](https://pure.tue.nl/ws/portalfiles/portal/370207745/DBP25-Conference_Paper_Final.pdf)

## Citation

Donkers, A. J. A., & Petrova, E. (2025). *Retrieval of Digital Regulations using Neuro-Symbolic AI with GraphRAG*. In S. Fischer, J. Fauth, H. Urban, & C. Schranz (Eds.), *Proceedings of the Digital Building Permit Conference 2025*. Springer Nature.

---

You made it to the end of the notebook. The main FireBIM project and its software components are documented in **ITEA4 22003 FireBIM D2.4 – Requirement Generator**.